# PySpark + Iceberg + Nessie Demo

This notebook demonstrates how to use PySpark with Apache Iceberg tables and Project Nessie catalog in a containerized environment.

## Architecture
- **PySpark**: Distributed data processing
- **Apache Iceberg**: Table format with ACID transactions
- **Project Nessie**: Git-like data versioning catalog
- **MinIO**: S3-compatible object storage

## Prerequisites
Make sure your Docker environment is running:
```bash
make start  # Start all services
make status # Check service health
```

In [1]:
# Import required libraries
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, when, lit
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)

print("✅ Imports successful!")
print(f"Python version: {sys.version}")

# Set environment variables for better output
os.environ["PYSPARK_SUBMIT_ARGS"] = "--jars /opt/spark/jars-custom/*.jar pyspark-shell"

✅ Imports successful!
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]


In [12]:
# Create SparkSession with Iceberg and Nessie configuration
spark = (
    SparkSession.builder.appName("IcebergNessieDemo")
    .master("spark://spark-master:7077")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config(
        "spark.sql.catalog.nessie.catalog-impl",
        "org.apache.iceberg.nessie.NessieCatalog",
    )
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

print("✅ SparkSession created successfully!")
print(f"Spark version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")


# Set default catalog
spark.sql("USE nessie")

✅ SparkSession created successfully!
Spark version: 3.5.0
Application ID: local-1755926827746


Py4JJavaError: An error occurred while calling o52.sql.
: java.lang.NoSuchMethodError: 'scala.collection.immutable.ArraySeq scala.runtime.ScalaRunTime$.wrapRefArray(java.lang.Object[])'
	at org.apache.spark.sql.catalyst.optimizer.ReplaceStaticInvoke$.$anonfun$apply$1(ReplaceStaticInvoke.scala:51)
	at org.apache.spark.sql.catalyst.optimizer.ReplaceStaticInvoke$.$anonfun$apply$1$adapted(ReplaceStaticInvoke.scala:51)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transformDownWithPruning(TreeNode.scala:457)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.org$apache$spark$sql$catalyst$plans$logical$AnalysisHelper$$super$transformDownWithPruning(LogicalPlan.scala:32)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.transformDownWithPruning(AnalysisHelper.scala:267)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.transformDownWithPruning$(AnalysisHelper.scala:263)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.transformDownWithPruning(L

In [11]:
# Create namespace (database) for our demo
# spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.unnest_master location 's3a://unnest/unnest_master'")

# Verify namespace creation
print("📁 Available namespaces:")
# spark.sql("SHOW NAMESPACES").show(5,False)

# Check current catalog and database
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current database: {spark.catalog.currentDatabase()}")

# Switch to our demo namespace
# spark.sql("USE nessie.unnest_master")

📁 Available namespaces:
Current catalog: spark_catalog
Current database: default


In [21]:
# spark.sql("DROP TABLE IF EXISTS events_master.employees")
# spark.sql("DROP NAMESPACE IF EXISTS nessie.events_master")
# spark.sql("DROP DATABASE IF EXISTS nessie.events_master")

DataFrame[]

In [24]:

print("📁 Available namespaces:")
spark.sql("SHOW NAMESPACES").show()

# Check current catalog and database
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current database: {spark.catalog.currentDatabase()}")


📁 Available namespaces:
+-------------+
|    namespace|
+-------------+
|unnest_master|
|         demo|
+-------------+

Current catalog: nessie
Current database: unnest_master


In [25]:
# Create sample employee data
schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),
        StructField("salary", DoubleType(), True),
        StructField("department", StringType(), True),
        # StructField("created_at", TimestampType(), True),
    ]
)

# Sample employee data
employees_data = [
    (1, "John Doe", 30, 75000.0, "Engineering"),
    (2, "Jane Smith", 25, 65000.0, "Marketing"),
    (3, "Bob Johnson", 35, 85000.0, "Engineering"),
    (4, "Alice Brown", 28, 70000.0, "Sales"),
    (5, "Charlie Wilson", 32, 80000.0, "Engineering"),
    (6, "Diana Garcia", 29, 72000.0, "Marketing"),
    (7, "Frank Miller", 31, 78000.0, "Sales"),
]

# Create DataFrame
df = spark.createDataFrame(employees_data, schema)
df = df.withColumn("created_at", current_timestamp())

print("🎯 Sample employee data created:")
df.show()
print(f"📊 Total records: {df.count()}")
print(f"📋 Schema:")
df.printSchema()

🎯 Sample employee data created:
+---+--------------+---+-------+-----------+--------------------+
| id|          name|age| salary| department|          created_at|
+---+--------------+---+-------+-----------+--------------------+
|  1|      John Doe| 30|75000.0|Engineering|2025-07-16 01:01:...|
|  2|    Jane Smith| 25|65000.0|  Marketing|2025-07-16 01:01:...|
|  3|   Bob Johnson| 35|85000.0|Engineering|2025-07-16 01:01:...|
|  4|   Alice Brown| 28|70000.0|      Sales|2025-07-16 01:01:...|
|  5|Charlie Wilson| 32|80000.0|Engineering|2025-07-16 01:01:...|
|  6|  Diana Garcia| 29|72000.0|  Marketing|2025-07-16 01:01:...|
|  7|  Frank Miller| 31|78000.0|      Sales|2025-07-16 01:01:...|
+---+--------------+---+-------+-----------+--------------------+

📊 Total records: 7
📋 Schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: double (nullable = true)
 |-- department: string (nullable = true)
 |-- created_at: tim

In [29]:
# Create Iceberg table
print("🏗️ Creating Iceberg table 'employees'...")

df.write.format("iceberg").mode("overwrite").saveAsTable("unnest_master.employees")

print("✅ Iceberg table 'employees' created successfully!")

# Verify table creation
print("\n📋 Tables in current namespace:")
spark.sql("SHOW TABLES").show()

# Check table properties
print("\n🔍 Table properties:")
spark.sql("SHOW TBLPROPERTIES employees").show(truncate=False)

🏗️ Creating Iceberg table 'employees'...
✅ Iceberg table 'employees' created successfully!

📋 Tables in current namespace:
+-------------+---------+-----------+
|    namespace|tableName|isTemporary|
+-------------+---------+-----------+
|unnest_master|employees|      false|
+-------------+---------+-----------+


🔍 Table properties:
+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |394901709814938577                                              |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.e

In [32]:
# Read data back from Iceberg table
print("📖 Reading data from Iceberg table:")
employees_df = spark.sql("SELECT * FROM unnest_master.employees")
employees_df.show(5,False)

📖 Reading data from Iceberg table:
+---+--------------+---+-------+-----------+--------------------------+
|id |name          |age|salary |department |created_at                |
+---+--------------+---+-------+-----------+--------------------------+
|1  |John Doe      |30 |75000.0|Engineering|2025-07-16 01:09:42.855732|
|2  |Jane Smith    |25 |65000.0|Marketing  |2025-07-16 01:09:42.855732|
|3  |Bob Johnson   |35 |85000.0|Engineering|2025-07-16 01:09:42.855732|
|4  |Alice Brown   |28 |70000.0|Sales      |2025-07-16 01:09:42.855732|
|5  |Charlie Wilson|32 |80000.0|Engineering|2025-07-16 01:09:42.855732|
+---+--------------+---+-------+-----------+--------------------------+
only showing top 5 rows



In [27]:
# Read data back from Iceberg table
print("📖 Reading data from Iceberg table:")
employees_df = spark.sql("SELECT * FROM employees")
employees_df.show()

# Basic DataFrame operations
print("\n🔍 Filter: Engineering department employees")
engineering_df = employees_df.filter(col("department") == "Engineering")
engineering_df.show()

print("\n📊 Group by department with average salary:")
dept_stats = (
    employees_df.groupBy("department")
    .agg({"salary": "avg", "age": "avg", "*": "count"})
    .withColumnRenamed("avg(salary)", "avg_salary")
    .withColumnRenamed("avg(age)", "avg_age")
    .withColumnRenamed("count(1)", "employee_count")
)

dept_stats.show()

print("\n💰 High earners (salary > 75000):")
high_earners = employees_df.filter(col("salary") > 75000).orderBy(col("salary").desc())
high_earners.show()

📖 Reading data from Iceberg table:
+---+--------------+---+-------+-----------+--------------------+
| id|          name|age| salary| department|          created_at|
+---+--------------+---+-------+-----------+--------------------+
|  1|      John Doe| 30|75000.0|Engineering|2025-07-16 01:01:...|
|  2|    Jane Smith| 25|65000.0|  Marketing|2025-07-16 01:01:...|
|  3|   Bob Johnson| 35|85000.0|Engineering|2025-07-16 01:01:...|
|  4|   Alice Brown| 28|70000.0|      Sales|2025-07-16 01:01:...|
|  5|Charlie Wilson| 32|80000.0|Engineering|2025-07-16 01:01:...|
|  6|  Diana Garcia| 29|72000.0|  Marketing|2025-07-16 01:01:...|
|  7|  Frank Miller| 31|78000.0|      Sales|2025-07-16 01:01:...|
+---+--------------+---+-------+-----------+--------------------+


🔍 Filter: Engineering department employees
+---+--------------+---+-------+-----------+--------------------+
| id|          name|age| salary| department|          created_at|
+---+--------------+---+-------+-----------+------------------

In [34]:
# Add new employees to the table
print("➕ Adding new employees...")

new_employees_data = [
    (8, "Sophie Turner", 27, 68000.0, "Marketing"),
    (9, "Michael Chen", 33, 82000.0, "Engineering"),
    (10, "Sarah Davis", 26, 71000.0, "Sales"),
]

new_df = spark.createDataFrame(new_employees_data, schema)
new_df = new_df.withColumn("created_at", current_timestamp())

# Append to existing table
new_df.write.format("iceberg").mode("append").saveAsTable("employees")

print("✅ New employees added!")

# Verify updated data
print(
    f"\n📊 Total employees now: {spark.sql('SELECT COUNT(*) as count FROM employees').collect()[0]['count']}"
)
spark.sql("SELECT * FROM employees ORDER BY id").show()

# Schema evolution example - add a new column
print("\n🔧 Adding email column to existing table...")
spark.sql("ALTER TABLE employees ADD COLUMN email STRING")

print("✅ Schema evolved! New schema:")
spark.sql("DESCRIBE employees").show()

➕ Adding new employees...


AnalysisException: [INCOMPATIBLE_DATA_FOR_TABLE.CANNOT_FIND_DATA] Cannot write incompatible data for the table `nessie`.`unnest_master`.`employees`: Cannot find data for the output column `email`.

In [35]:
# Explore Iceberg table history and time travel
print("📚 Table history:")
history_df = spark.sql("SELECT * FROM unnest_master.employees")
history_df.show(truncate=False)

📚 Table history:
+---+--------------+---+-------+-----------+--------------------------+-----+
|id |name          |age|salary |department |created_at                |email|
+---+--------------+---+-------+-----------+--------------------------+-----+
|8  |Sophie Turner |27 |68000.0|Marketing  |2025-07-16 01:17:08.462216|NULL |
|9  |Michael Chen  |33 |82000.0|Engineering|2025-07-16 01:17:08.462216|NULL |
|10 |Sarah Davis   |26 |71000.0|Sales      |2025-07-16 01:17:08.462216|NULL |
|1  |John Doe      |30 |75000.0|Engineering|2025-07-16 01:09:42.855732|NULL |
|2  |Jane Smith    |25 |65000.0|Marketing  |2025-07-16 01:09:42.855732|NULL |
|3  |Bob Johnson   |35 |85000.0|Engineering|2025-07-16 01:09:42.855732|NULL |
|4  |Alice Brown   |28 |70000.0|Sales      |2025-07-16 01:09:42.855732|NULL |
|5  |Charlie Wilson|32 |80000.0|Engineering|2025-07-16 01:09:42.855732|NULL |
|6  |Diana Garcia  |29 |72000.0|Marketing  |2025-07-16 01:09:42.855732|NULL |
|7  |Frank Miller  |31 |78000.0|Sales      |202

In [ ]:
# Explore Iceberg table history and time travel
print("📚 Table history:")
history_df = spark.sql("SELECT * FROM employees.history")
history_df.show(truncate=False)

print("\n📂 Table snapshots:")
snapshots_df = spark.sql("SELECT * FROM employees.snapshots")
snapshots_df.select("snapshot_id", "timestamp_ms", "summary").show(truncate=False)

print("\n📁 Table files:")
files_df = spark.sql("SELECT * FROM employees.files")
print(f"Number of data files: {files_df.count()}")
files_df.select("file_path", "file_format", "record_count", "file_size_in_bytes").show(
    truncate=False
)

# Time travel example (query specific snapshot)
print("\n⏰ Time travel - querying first snapshot:")
snapshots = spark.sql(
    "SELECT snapshot_id FROM employees.snapshots ORDER BY committed_at"
).collect()
if len(snapshots) > 0:
    first_snapshot = snapshots[0]["snapshot_id"]
    time_travel_df = spark.sql(
        f"SELECT * FROM employees VERSION AS OF {first_snapshot}"
    )
    print(f"Records in first snapshot: {time_travel_df.count()}")
    time_travel_df.show()

In [ ]:
# Nessie branching and versioning
print("🌿 Working with Nessie branches...")

# Create a new branch for experimental changes
spark.sql("CREATE BRANCH feature_branch IN nessie FROM main")
print("✅ Created feature branch")

# List all branches
print("\n📋 Available branches:")
spark.sql("SHOW BRANCHES IN nessie").show()

# Switch to feature branch
spark.sql("USE BRANCH feature_branch IN nessie")
print("🔄 Switched to feature_branch")

# Make changes in the feature branch
print("\n🔧 Making changes in feature branch...")
spark.sql(
    """
    UPDATE employees 
    SET salary = salary * 1.1 
    WHERE department = 'Engineering'
"""
)

print("💰 Updated Engineering salaries (+10%):")
spark.sql(
    "SELECT name, department, salary FROM employees WHERE department = 'Engineering'"
).show()

# Switch back to main to see original data
spark.sql("USE BRANCH main IN nessie")
print("\n📊 Back on main branch - original salaries:")
spark.sql(
    "SELECT name, department, salary FROM employees WHERE department = 'Engineering'"
).show()

In [ ]:
# Advanced SQL queries and analytics
print("📊 Advanced Analytics Examples")

# Create a temporary view for complex queries
employees_df.createOrReplaceTempView("employees_view")

# Window function example
print("\n🏆 Salary ranking within departments:")
rank_query = """
SELECT 
    name,
    department,
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as salary_rank,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as row_num
FROM employees_view
ORDER BY department, salary_rank
"""
spark.sql(rank_query).show()

# Age distribution analysis
print("\n📈 Age distribution by department:")
age_analysis = """
SELECT 
    department,
    COUNT(*) as employee_count,
    AVG(age) as avg_age,
    MIN(age) as min_age,
    MAX(age) as max_age,
    STDDEV(age) as age_stddev
FROM employees_view
GROUP BY department
ORDER BY avg_age DESC
"""
spark.sql(age_analysis).show()

# Salary percentiles
print("\n💰 Salary percentiles:")
percentile_query = """
SELECT 
    department,
    PERCENTILE_APPROX(salary, 0.25) as salary_25th,
    PERCENTILE_APPROX(salary, 0.5) as salary_median,
    PERCENTILE_APPROX(salary, 0.75) as salary_75th
FROM employees_view
GROUP BY department
ORDER BY salary_median DESC
"""
spark.sql(percentile_query).show()

In [ ]:
# Performance monitoring and table maintenance
print("⚡ Performance and Maintenance Operations")

# Check table statistics
print("\n📊 Table statistics:")
spark.sql("ANALYZE TABLE employees COMPUTE STATISTICS").show()

# Show table size and file information
print("\n💾 Storage information:")
storage_info = spark.sql(
    """
    SELECT 
        'Total Files' as metric,
        COUNT(*) as value
    FROM employees.files
    UNION ALL
    SELECT 
        'Total Size (MB)' as metric,
        ROUND(SUM(file_size_in_bytes) / (1024 * 1024), 2) as value
    FROM employees.files
    UNION ALL
    SELECT 
        'Total Records' as metric,
        SUM(record_count) as value
    FROM employees.files
"""
)
storage_info.show()

# Show partition information if any
print("\n📂 Data organization:")
spark.sql("SHOW PARTITIONS employees").show()

print("\n🧹 Cleanup operations available:")
print("- CALL nessie.system.rewrite_data_files('employees') -- Compact small files")
print(
    "- CALL nessie.system.expire_snapshots('employees', TIMESTAMP '2024-01-01 00:00:00') -- Remove old snapshots"
)
print("- CALL nessie.system.remove_orphan_files('employees') -- Clean orphaned files")

In [ ]:
# Summary and cleanup
print("🎉 Demo Summary")
print("=" * 50)

# Final statistics
final_count = spark.sql("SELECT COUNT(*) as count FROM employees").collect()[0]["count"]
branches_count = spark.sql("SHOW BRANCHES IN nessie").count()
snapshots_count = spark.sql(
    "SELECT COUNT(*) as count FROM employees.snapshots"
).collect()[0]["count"]

print(f"✅ Total employees created: {final_count}")
print(f"🌿 Total branches: {branches_count}")
print(f"📸 Total snapshots: {snapshots_count}")
print(f"🏗️ Tables created: employees")
print(f"🗂️ Namespaces: demo")

print("\n🔍 What we demonstrated:")
print("- ✅ PySpark session with Iceberg extensions")
print("- ✅ Iceberg table creation and data insertion")
print("- ✅ Schema evolution (adding columns)")
print("- ✅ Time travel queries")
print("- ✅ Nessie branching and versioning")
print("- ✅ Advanced SQL analytics")
print("- ✅ Performance monitoring")

print("\n🎯 Next steps:")
print("- Try querying from Trino: make query")
print("- Explore more Nessie features")
print("- Create more complex data pipelines")
print("- Set up automated table maintenance")

# Optional: Stop SparkSession (comment out if you want to keep it running)
# print("\\n🛑 Stopping SparkSession...")
# spark.stop()
# print("✅ SparkSession stopped")